In [1]:
# imports
import json
import pandas as pd
from pathlib import Path

# chemins entree sortie
INTERIM_PATH = Path("../../medical_dataset/interim/dialogues_clean.parquet")
PROCESSED_DIR = Path("../../medical_dataset/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# parametres reproductibilite et split
SEED = 42
VAL_SIZE = 0.05
TEST_SIZE = 0.05

In [3]:
# chargement des donnees nettoyees
df = pd.read_parquet(INTERIM_PATH)

print("Shape:", df.shape)
print("Colonnes:", df.columns.tolist())

df.head(3)

Shape: (245955, 3)
Colonnes: ['Description', 'Patient', 'Doctor']


,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...


In [4]:
# prompt systeme qui cadre le comportement de l assistant medical
SYSTEM_PROMPT = (
    "You are a helpful and empathetic medical assistant. "
    "You provide clear, accurate and responsible health information. "
    "You always remind the user to consult a qualified healthcare professional "
    "for a real diagnosis or emergency."
)

In [5]:
# construction d un exemple au format conversationnel messages
def build_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["Patient"].strip()},
        {"role": "assistant", "content": row["Doctor"].strip()},
    ]

df["messages"] = df.apply(build_messages, axis=1)
df[["messages"]].head(2).to_dict("records")[0]

{'messages': [{'role': 'system',
   'content': 'You are a helpful and empathetic medical assistant. You provide clear, accurate and responsible health information. You always remind the user to consult a qualified healthcare professional for a real diagnosis or emergency.'},
  {'role': 'user',
   'content': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?'},
  {'role': 'assistant',
   'content': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'}]}

In [6]:
# rendu texte lisible via un template chatml pour inspection ou usage direct
def render_text(messages):
    parts = []
    for m in messages:
        parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>")
    return "\n".join(parts)

df["text"] = df["messages"].apply(render_text)

print(df["text"].iloc[0][:600])

<|im_start|>system
You are a helpful and empathetic medical assistant. You provide clear, accurate and responsible health information. You always remind the user to consult a qualified healthcare professional for a real diagnosis or emergency.<|im_end|>
<|im_start|>user
Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for annular bulging and tear?<|im_end|>
<|im_start|>assistant
Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further inform


In [7]:
# controle des longueurs en tokens approximes par les mots
df["n_words"] = df["text"].str.split().str.len()

print(df["n_words"].describe().round(0))

count    245955.0
mean        206.0
std          73.0
min          47.0
25%         158.0
50%         192.0
75%         236.0
max        1029.0
Name: n_words, dtype: float64


In [8]:
# split train val test par simple melange reproductible
shuffled = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n = len(shuffled)
n_test = int(n * TEST_SIZE)
n_val = int(n * VAL_SIZE)

test_df = shuffled.iloc[:n_test].reset_index(drop=True)
val_df = shuffled.iloc[n_test:n_test + n_val].reset_index(drop=True)
train_df = shuffled.iloc[n_test + n_val:].reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 221361 | Val: 12297 | Test: 12297


In [10]:
# export jsonl format standard pour le sft une ligne un exemple messages
def save_jsonl(frame, path):
    with open(path, "w", encoding="utf-8") as f:
        for messages in frame["messages"]:
            f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")

save_jsonl(train_df, PROCESSED_DIR / "train.jsonl")
save_jsonl(val_df, PROCESSED_DIR / "validation.jsonl")
save_jsonl(test_df, PROCESSED_DIR / "test.jsonl")

In [11]:
# export parquet egalement pratique pour datasets load_dataset
cols = ["Description", "Patient", "Doctor", "messages", "text"]

train_df[cols].to_parquet(PROCESSED_DIR / "train.parquet")
val_df[cols].to_parquet(PROCESSED_DIR / "validation.parquet")
test_df[cols].to_parquet(PROCESSED_DIR / "test.parquet")

## Résumé

- **Entrée** : `medical_dataset/interim/dialogues_clean.parquet`
- **Format** : conversations `system` / `user` / `assistant` (clé `messages`) prêtes pour un SFT LoRA (TRL `SFTTrainer`, chat template)
- **Sorties** dans `medical_dataset/processed` :
  - `train.jsonl` / `validation.jsonl` / `test.jsonl` (format SFT standard)
  - `train.parquet` / `validation.parquet` / `test.parquet` (rechargeables via `datasets.load_dataset`)
- **Split** : 90 % train / 5 % validation / 5 % test (`seed=42`)